# LoViF 2026 - StableCodec FT8 Downscale/Upscale Clean Submission

This notebook tests one clean architecture:

```text
source image -> downsample -> StableCodec ft8 real bitstream -> decode LR -> upscale to original size
```

Cleanliness rules:

- uses only `stablecodec_ft8.pkl`;
- no ft16/ft24/ft32 fallback;
- no source-image color fix;
- no post-decode original-image statistics;
- no fake, proxy, estimated-size, or diagnostic `.bin` files;
- every submitted `.bin` contains a counted metadata header plus the real StableCodec entropy stream.

The header is needed because the decoder must know the original target size, selected scale,
downsampler, and upscaler. Header bytes are counted in BPP.

Output:

```text
/kaggle/working/lovif_stablecodec_ft8_downscale_submission.zip
```


In [ ]:
from pathlib import Path

EVALUATOR_SOURCE = '#!/usr/bin/env python3\n"""Evaluate a LoViF ultra-low bitrate submission locally.\n\nUsage:\n    python scripts/evaluate_submission.py filtered_dataset/dataset_val lovif_pretrained_prior_submission\n    python scripts/evaluate_submission.py filtered_dataset/dataset_val lovif_pretrained_prior_submission.zip\n\nThe submission must contain:\n    reconstructed/<same image names as dataset_val>\n    bitstream/<same stem>.bin\n    readme.txt\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport zipfile\nfrom pathlib import Path\n\nfrom PIL import Image, ImageOps\n\n\nIMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}\nBPP_LIMIT = 0.008\nMETRIC_PROFILES = ("codabench-observed", "pyiqa-pretrained")\n\n\ndef list_images(folder: Path) -> list[Path]:\n    return sorted(\n        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS],\n        key=lambda p: p.name,\n    )\n\n\ndef open_rgb(path: Path) -> Image.Image:\n    with Image.open(path) as img:\n        return ImageOps.exif_transpose(img).convert("RGB")\n\n\ndef prepare_submission_path(path: Path, tmp_dir: Path) -> Path:\n    if path.is_dir():\n        return path\n    if path.is_file() and path.suffix.lower() == ".zip":\n        out = tmp_dir / "submission"\n        out.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(path, "r") as zf:\n            zf.extractall(out)\n        return out\n    raise FileNotFoundError(f"Submission path is neither a folder nor a zip: {path}")\n\n\ndef require_pyiqa(install_missing: bool):\n    try:\n        import pyiqa  # type: ignore\n        import torch  # type: ignore\n    except Exception as exc:\n        if install_missing:\n            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyiqa"])\n            import pyiqa  # type: ignore\n            import torch  # type: ignore\n            return pyiqa, torch\n        raise RuntimeError(\n            "This evaluator needs pyiqa and torch for the official metrics. "\n            "Install them with: pip install pyiqa torch torchvision\\n"\n            "Then run WITHOUT --structure-only. You can also pass --install-missing."\n        ) from exc\n    return pyiqa, torch\n\n\ndef create_metrics(device: str, install_missing: bool, metric_profile: str, perceptual_seed: int | None):\n    pyiqa, torch = require_pyiqa(install_missing)\n    if device == "auto":\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n\n    metrics = {\n        "device": device,\n        "metric_profile": metric_profile,\n        "perceptual_seed": perceptual_seed,\n        "psnr": pyiqa.create_metric("psnr", device=device),\n        "ms_ssim": pyiqa.create_metric("ms_ssim", device=device),\n    }\n    if metric_profile == "pyiqa-pretrained":\n        metrics["lpips"] = pyiqa.create_metric("lpips", device=device)\n        metrics["dists"] = pyiqa.create_metric("dists", device=device)\n        metrics["metric_note"] = "Standard pyiqa pretrained LPIPS/DISTS. This did not match the observed Codabench row."\n    elif metric_profile == "codabench-observed":\n        if perceptual_seed is not None:\n            torch.manual_seed(perceptual_seed)\n        metrics["lpips"] = pyiqa.create_metric("lpips", device=device, pretrained=False)\n        metrics["dists"] = pyiqa.create_metric("dists", device=device, pretrained=False)\n        metrics["metric_note"] = (\n            "Reverse-engineered from the observed Codabench leaderboard: PSNR/MS-SSIM use pyiqa defaults; "\n            "LPIPS/DISTS use pyiqa with pretrained=False to match the leaderboard behavior."\n        )\n    else:\n        raise ValueError(f"Unknown metric profile: {metric_profile}")\n\n    return metrics\n\n\ndef metric_value(metric, rec_path: Path, gt_path: Path) -> float:\n    value = metric(str(rec_path), str(gt_path))\n    return float(value.detach().cpu().squeeze().item())\n\n\ndef validate_structure(gt_paths: list[Path], submission_dir: Path) -> tuple[Path, Path, list[str]]:\n    rec_dir = submission_dir / "reconstructed"\n    bit_dir = submission_dir / "bitstream"\n    errors: list[str] = []\n\n    if not rec_dir.is_dir():\n        errors.append(f"Missing folder: {rec_dir}")\n    if not bit_dir.is_dir():\n        errors.append(f"Missing folder: {bit_dir}")\n    if not (submission_dir / "readme.txt").exists():\n        errors.append(f"Missing file: {submission_dir / \'readme.txt\'}")\n\n    if errors:\n        return rec_dir, bit_dir, errors\n\n    for gt_path in gt_paths:\n        rec_path = rec_dir / gt_path.name\n        bit_path = bit_dir / f"{gt_path.stem}.bin"\n        if not rec_path.exists():\n            errors.append(f"Missing reconstructed image: reconstructed/{gt_path.name}")\n            continue\n        if not bit_path.exists():\n            errors.append(f"Missing bitstream: bitstream/{gt_path.stem}.bin")\n            continue\n\n        gt_img = open_rgb(gt_path)\n        rec_img = open_rgb(rec_path)\n        if rec_img.size != gt_img.size:\n            errors.append(\n                f"Resolution mismatch for {gt_path.name}: "\n                f"reconstructed={rec_img.size}, ground_truth={gt_img.size}"\n            )\n\n    return rec_dir, bit_dir, errors\n\n\ndef evaluate_structure_only(dataset_val: Path, submission: Path, max_images: int | None) -> dict:\n    gt_paths = list_images(dataset_val)\n    if max_images is not None:\n        gt_paths = gt_paths[:max_images]\n    if not gt_paths:\n        raise FileNotFoundError(f"No images found in {dataset_val}")\n\n    with tempfile.TemporaryDirectory(prefix="lovif_eval_") as tmp:\n        submission_dir = prepare_submission_path(submission, Path(tmp))\n        rec_dir, bit_dir, errors = validate_structure(gt_paths, submission_dir)\n        if errors:\n            raise AssertionError("\\n".join(errors[:50]))\n\n        rows = []\n        total_bytes = 0\n        total_pixels = 0\n        for gt_path in gt_paths:\n            bit_path = bit_dir / f"{gt_path.stem}.bin"\n            gt_img = open_rgb(gt_path)\n            pixels = gt_img.size[0] * gt_img.size[1]\n            bit_bytes = bit_path.stat().st_size\n            total_bytes += bit_bytes\n            total_pixels += pixels\n            rows.append(\n                {\n                    "file": gt_path.name,\n                    "width": gt_img.size[0],\n                    "height": gt_img.size[1],\n                    "pixels": pixels,\n                    "bitstream_bytes": bit_bytes,\n                    "bpp": 8.0 * bit_bytes / pixels,\n                }\n            )\n\n        avg_bpp = 8.0 * total_bytes / total_pixels\n        return {\n            "mode": "structure_only",\n            "note": "This is NOT the full Codabench/leaderboard evaluation. Run without --structure-only for PSNR/MS-SSIM/LPIPS/DISTS/Final Score.",\n            "summary": {\n                "image_count": len(rows),\n                "total_bitstream_bytes": total_bytes,\n                "total_pixels": total_pixels,\n                "bpp": avg_bpp,\n                "bpp_limit": BPP_LIMIT,\n                "valid_bpp": avg_bpp <= BPP_LIMIT,\n            },\n            "leaderboard": {\n                "Final Score": None,\n                "PSNR": None,\n                "MS-SSIM": None,\n                "LPIPS": None,\n                "DISTS": None,\n                "BPP": avg_bpp,\n                "Runtime per image [s]": None,\n                "CPU [1]/ GPU [0]": None,\n            },\n            "rows": rows,\n        }\n\n\ndef evaluate(\n    dataset_val: Path,\n    submission: Path,\n    device: str,\n    max_images: int | None,\n    install_missing: bool,\n    metric_profile: str,\n    perceptual_seed: int | None,\n) -> dict:\n    gt_paths = list_images(dataset_val)\n    if max_images is not None:\n        gt_paths = gt_paths[:max_images]\n    if not gt_paths:\n        raise FileNotFoundError(f"No images found in {dataset_val}")\n\n    with tempfile.TemporaryDirectory(prefix="lovif_eval_") as tmp:\n        submission_dir = prepare_submission_path(submission, Path(tmp))\n        rec_dir, bit_dir, errors = validate_structure(gt_paths, submission_dir)\n        if errors:\n            raise AssertionError("\\n".join(errors[:50]))\n\n        metrics = create_metrics(device, install_missing, metric_profile, perceptual_seed)\n        rows = []\n        total_bytes = 0\n        total_pixels = 0\n        start = time.perf_counter()\n\n        for idx, gt_path in enumerate(gt_paths, start=1):\n            rec_path = rec_dir / gt_path.name\n            bit_path = bit_dir / f"{gt_path.stem}.bin"\n\n            gt_img = open_rgb(gt_path)\n            pixels = gt_img.size[0] * gt_img.size[1]\n            bit_bytes = bit_path.stat().st_size\n            total_bytes += bit_bytes\n            total_pixels += pixels\n\n            psnr = metric_value(metrics["psnr"], rec_path, gt_path)\n            ms_ssim = metric_value(metrics["ms_ssim"], rec_path, gt_path)\n            lpips = metric_value(metrics["lpips"], rec_path, gt_path)\n            dists = metric_value(metrics["dists"], rec_path, gt_path)\n\n            rows.append(\n                {\n                    "file": gt_path.name,\n                    "width": gt_img.size[0],\n                    "height": gt_img.size[1],\n                    "pixels": pixels,\n                    "bitstream_bytes": bit_bytes,\n                    "bpp": 8.0 * bit_bytes / pixels,\n                    "psnr": psnr,\n                    "ms_ssim": ms_ssim,\n                    "lpips": lpips,\n                    "dists": dists,\n                }\n            )\n\n            print(\n                f"[{idx:03d}/{len(gt_paths):03d}] {gt_path.name} "\n                f"PSNR={psnr:.4f} MS-SSIM={ms_ssim:.4f} "\n                f"LPIPS={lpips:.4f} DISTS={dists:.4f} "\n                f"BPP={rows[-1][\'bpp\']:.6f}",\n                flush=True,\n            )\n\n        elapsed = time.perf_counter() - start\n        avg_bpp = 8.0 * total_bytes / total_pixels\n\n        mean_psnr = sum(r["psnr"] for r in rows) / len(rows)\n        mean_ms_ssim = sum(r["ms_ssim"] for r in rows) / len(rows)\n        mean_lpips = sum(r["lpips"] for r in rows) / len(rows)\n        mean_dists = sum(r["dists"] for r in rows) / len(rows)\n        # Same as Codabench page:\n        # Final_Score = PSNR + 10 * MS_SSIM + 20 * (1 - LPIPS) + 25 * (1 - DISTS)\n        final_score = mean_psnr + 10.0 * mean_ms_ssim - 20.0 * mean_lpips - 25.0 * mean_dists + 45.0\n        platform_flag = 0 if metrics["device"] == "cuda" else 1\n\n        return {\n            "mode": "full_leaderboard",\n            "summary": {\n                "image_count": len(rows),\n                "device": metrics["device"],\n                "metric_profile": metrics["metric_profile"],\n                "metric_note": metrics["metric_note"],\n                "perceptual_seed": metrics["perceptual_seed"],\n                "total_bitstream_bytes": total_bytes,\n                "total_pixels": total_pixels,\n                "bpp": avg_bpp,\n                "bpp_limit": BPP_LIMIT,\n                "valid_bpp": avg_bpp <= BPP_LIMIT,\n                "psnr": mean_psnr,\n                "ms_ssim": mean_ms_ssim,\n                "lpips": mean_lpips,\n                "dists": mean_dists,\n                "final_score": final_score,\n                "elapsed_s": elapsed,\n                "runtime_per_image_s": elapsed / len(rows),\n                "cpu_1_gpu_0": platform_flag,\n            },\n            "leaderboard": {\n                "Final Score": final_score,\n                "PSNR": mean_psnr,\n                "MS-SSIM": mean_ms_ssim,\n                "LPIPS": mean_lpips,\n                "DISTS": mean_dists,\n                "BPP": avg_bpp,\n                "Runtime per image [s]": elapsed / len(rows),\n                "CPU [1]/ GPU [0]": platform_flag,\n            },\n            "rows": rows,\n        }\n\n\ndef write_outputs(result: dict, output_json: Path | None, output_csv: Path | None) -> None:\n    if output_json:\n        output_json.parent.mkdir(parents=True, exist_ok=True)\n        output_json.write_text(json.dumps(result, indent=2), encoding="utf-8")\n\n    if output_csv:\n        output_csv.parent.mkdir(parents=True, exist_ok=True)\n        rows = result["rows"]\n        if rows:\n            with output_csv.open("w", newline="", encoding="utf-8") as f:\n                writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))\n                writer.writeheader()\n                writer.writerows(rows)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Evaluate a LoViF development submission locally.")\n    parser.add_argument("dataset_val", type=Path, help="Path to dataset_val ground-truth image folder.")\n    parser.add_argument("submission", type=Path, help="Path to submission folder or submission zip.")\n    parser.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"], help="Metric device.")\n    parser.add_argument(\n        "--metric-profile",\n        default="codabench-observed",\n        choices=METRIC_PROFILES,\n        help=(\n            "Metric implementation profile. Use codabench-observed for the current leaderboard behavior; "\n            "use pyiqa-pretrained for standard pretrained LPIPS/DISTS."\n        ),\n    )\n    parser.add_argument(\n        "--perceptual-seed",\n        type=int,\n        default=0,\n        help="Torch seed used before constructing codabench-observed LPIPS/DISTS. Ignored by pyiqa-pretrained.",\n    )\n    parser.add_argument("--max-images", type=int, default=None, help="Evaluate only the first N images.")\n    parser.add_argument(\n        "--structure-only",\n        action="store_true",\n        help="Only validate folders, filenames, resolutions, and BPP. Does not import pyiqa.",\n    )\n    parser.add_argument(\n        "--install-missing",\n        action="store_true",\n        help="Attempt to pip install pyiqa if it is missing.",\n    )\n    parser.add_argument("--json", type=Path, default=None, help="Output JSON path.")\n    parser.add_argument("--csv", type=Path, default=None, help="Output per-image CSV path.")\n    return parser.parse_args()\n\n\ndef main() -> int:\n    args = parse_args()\n    if args.json is None:\n        args.json = Path("local_structure_results.json" if args.structure_only else "local_eval_results.json")\n    if args.csv is None:\n        args.csv = Path("local_structure_per_image.csv" if args.structure_only else "local_eval_per_image.csv")\n    try:\n        if args.structure_only:\n            result = evaluate_structure_only(args.dataset_val, args.submission, args.max_images)\n        else:\n            result = evaluate(\n                args.dataset_val,\n                args.submission,\n                args.device,\n                args.max_images,\n                args.install_missing,\n                args.metric_profile,\n                args.perceptual_seed,\n            )\n        write_outputs(result, args.json, args.csv)\n    except Exception as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 1\n\n    summary = result["summary"]\n    print("\\nSummary")\n    if not args.structure_only:\n        leaderboard = result["leaderboard"]\n        print(f"  Metric profile       : {summary[\'metric_profile\']}")\n        print(f"  Final Score          : {leaderboard[\'Final Score\']:.4f}")\n        print(f"  PSNR                 : {leaderboard[\'PSNR\']:.4f}")\n        print(f"  MS-SSIM              : {leaderboard[\'MS-SSIM\']:.4f}")\n        print(f"  LPIPS                : {leaderboard[\'LPIPS\']:.4f}")\n        print(f"  DISTS                : {leaderboard[\'DISTS\']:.4f}")\n        print(f"  Runtime per image [s]: {leaderboard[\'Runtime per image [s]\']:.4f}")\n        print(f"  CPU [1]/ GPU [0]     : {leaderboard[\'CPU [1]/ GPU [0]\']}")\n    else:\n        print("  Mode        : structure-only, not full leaderboard metrics")\n    print(f"  BPP         : {summary[\'bpp\']:.6f} / {summary[\'bpp_limit\']}")\n    print(f"  BPP Valid   : {summary[\'valid_bpp\']}")\n    if not args.structure_only:\n        print(f"  Runtime/img : {summary[\'runtime_per_image_s\']:.4f} s")\n    print(f"  JSON        : {args.json}")\n    print(f"  CSV         : {args.csv}")\n    return 0 if summary["valid_bpp"] else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
Path("scripts").mkdir(parents=True, exist_ok=True)
Path("scripts/evaluate_submission.py").write_text(EVALUATOR_SOURCE, encoding="utf-8")
print("Wrote scripts/evaluate_submission.py")


In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import re
import shutil
import struct
import subprocess
import sys
import time
import zipfile
from contextlib import contextmanager
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Iterable

import numpy as np
from PIL import Image, ImageOps

Image.MAX_IMAGE_PIXELS = None

# =========================
# User Config
# =========================
BPP_LIMIT = 0.008
BPP_TARGET = 0.00799
SEED = 123
DEVICE = "cuda"
MAX_IMAGES = None  # set to an int for a quick smoke test

# ft8-only experiment. Keep this list small enough for Kaggle runtime.
# ft8 is higher bitrate than ft16, so these stronger downscales are used to fit <=0.008 effective bpp.
SCALES = [0.75, 0.70, 0.675, 0.65, 0.625, 0.60]
DOWNSAMPLERS = ["lanczos"]
UPSCALERS = ["lanczos"]

RUN_STRUCTURE_EVALUATOR = True
RUN_BOARD_FAITHFUL_SCORE = True
RUN_FULL_METRIC_SELECTION = True
SAVE_CANDIDATE_IMAGES = True
ENABLE_XFORMERS = False

# Important: keep disabled. StableCodec color_fix uses source-image
# statistics after decoding, which is not clean unless transmitted.
ALLOW_SOURCE_COLOR_FIX = False

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_WORKING.exists()
WORK = KAGGLE_WORKING if IS_KAGGLE else Path.cwd() / "ft8_downscale_work"
WORK.mkdir(parents=True, exist_ok=True)

LOVIF_DATASET_ROOT = Path("/kaggle/input/datasets/tonyironman099/lovif-2026-image-compression")
STABLECODEC_CHECKPOINT_INPUT_DIR = Path("/kaggle/input/datasets/mehedi052/stablecodec-checkpoints")

REPO_DIR = WORK / "StableCodec"
MODEL_DIR = WORK / "models"
CKPT_DIR = MODEL_DIR / "stablecodec_ckpts"
SD_TURBO_DIR = MODEL_DIR / "sd-turbo"
OUT_ROOT = WORK / "stablecodec_ft8_downscale_candidates"
FINAL_ROOT = WORK / "lovif_stablecodec_ft8_downscale_submission"
FINAL_ZIP = WORK / "lovif_stablecodec_ft8_downscale_submission.zip"

STABLECODEC_REPO_URL = "https://github.com/LuizScarlet/StableCodec.git"
GDRIVE_IDS = {
    "elic_official.pth": "1jUfYJdZd0-bYUsoOUWwEpI5t1MZYP3AP",
    "stablecodec_ft8.pkl": "1CVXLaVv48vM1Iy_Zqe3wcbJq_Lnh0vNw",
}
EXPECTED_MIN_BYTES = {
    "elic_official.pth": 160_000_000,
    "stablecodec_ft8.pkl": 430_000_000,
}

DATASET_ROOT_CANDIDATES = [
    LOVIF_DATASET_ROOT,
    Path("/kaggle/input/datasets/dipit099/lovif-2026-compression-algo"),
    Path("/kaggle/input/lovif-2026-image-compression"),
    Path("/kaggle/input"),
    Path("filtered_dataset"),
    Path("."),
]

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}
METHOD_IDS = {"bicubic": 0, "lanczos": 1}
ID_TO_METHOD = {v: k for k, v in METHOD_IDS.items()}
SCALE_IDS = {float(s): i for i, s in enumerate(SCALES)}

# Header is counted in BPP:
# magic(5), version(1), scale_id(1), down_id(1), up_id(1),
# original_h/w(4+4), lr_h/w(4+4).
HEADER_MAGIC = b"SCDS1"
HEADER_VERSION = 1
HEADER_STRUCT = ">5sBBBBIIII"
HEADER_LEN = struct.calcsize(HEADER_STRUCT)

CSV_COLUMNS = [
    "image_name", "checkpoint_name", "original_height", "original_width",
    "scale", "downsampled_height", "downsampled_width", "downsampler", "upscaler",
    "bitstream_bytes", "actual_bits", "nominal_lr_bpp", "effective_bpp",
    "bpp_budget_pass", "PSNR", "MS_SSIM", "LPIPS", "DISTS", "Final_Score",
    "encode_time_sec", "decode_time_sec", "upscale_time_sec", "total_time_sec",
]

def run(cmd: Iterable[object], cwd: Path | None = None, check: bool = True) -> int:
    cmd = [str(c) for c in cmd]
    print("\\n$ " + " ".join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")
    return result.returncode

def pip_install(packages: list[str]) -> None:
    run([sys.executable, "-m", "pip", "install", "-q", *packages], check=False)

def list_images(folder: Path) -> list[Path]:
    if not folder.exists():
        return []
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS],
        key=lambda p: p.name,
    )

def find_split(split: str) -> Path | None:
    for root in DATASET_ROOT_CANDIDATES:
        if not root.exists():
            continue
        direct = root / split
        if direct.is_dir() and list_images(direct):
            return direct
        for p in root.rglob(split):
            if p.is_dir() and list_images(p):
                return p
    return None

def open_rgb(path: Path) -> Image.Image:
    with Image.open(path) as img:
        return ImageOps.exif_transpose(img).convert("RGB")

def pil_resample(name: str) -> int:
    if name == "bicubic":
        return Image.Resampling.BICUBIC
    if name == "lanczos":
        return Image.Resampling.LANCZOS
    raise ValueError(f"Unsupported resize method: {name}")

def scale_tag(scale: float) -> str:
    return f"{scale:g}".replace(".", "p")

def setting_name(scale: float, downsampler: str) -> str:
    return f"ft8_s{scale_tag(scale)}_down_{downsampler}"

def candidate_stem(image_stem: str, scale: float, downsampler: str, upscaler: str) -> str:
    return f"{image_stem}__ckpt-stablecodec_ft8__scale-{scale_tag(scale)}__down-{downsampler}__up-{upscaler}"

def round_scaled_size(width: int, height: int, scale: float) -> tuple[int, int]:
    return max(1, int(round(width * scale))), max(1, int(round(height * scale)))

def image_pixels(path: Path) -> int:
    with Image.open(path) as img:
        w, h = img.size
    return w * h

def reset_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def write_csv(path: Path, rows: list[dict], columns: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            out = {}
            for key in columns:
                value = row.get(key)
                if isinstance(value, float):
                    out[key] = f"{value:.10f}"
                elif value is None:
                    out[key] = ""
                else:
                    out[key] = value
            writer.writerow(out)

TEST_DIR = find_split("dataset_test")
VAL_DIR = find_split("dataset_val")
INPUT_DIR = TEST_DIR or VAL_DIR
if INPUT_DIR is None:
    raise FileNotFoundError("Could not find dataset_test or dataset_val.")

input_paths = list_images(INPUT_DIR)
if MAX_IMAGES is not None:
    input_paths = input_paths[:MAX_IMAGES]
if not input_paths:
    raise RuntimeError(f"No input images found in {INPUT_DIR}")

total_pixels = sum(image_pixels(p) for p in input_paths)
byte_budget = math.floor(BPP_TARGET * total_pixels / 8.0)
absolute_byte_budget = math.floor(BPP_LIMIT * total_pixels / 8.0)

print("WORK:", WORK)
print("INPUT_DIR:", INPUT_DIR)
print("Images:", len(input_paths))
print("SCALES:", SCALES)
print("DOWNSAMPLERS:", DOWNSAMPLERS)
print("UPSCALERS:", UPSCALERS)
print("Total pixels:", total_pixels)
print("Selection byte budget:", byte_budget, "absolute 0.008 budget:", absolute_byte_budget)
print("Header bytes per image:", HEADER_LEN)


In [ ]:
packages = [
    "gdown",
    "huggingface_hub==0.25.0",
    "diffusers==0.25.1",
    "transformers==4.46.3",
    "accelerate==1.9.0",
    "peft",
    "omegaconf",
    "einops>=0.6.1",
    "timm==1.0.22",
    "compressai==1.2.6",
    "open-clip-torch>=2.20.0",
    "lpips",
    "DISTS_pytorch",
    "pytorch-msssim",
]
pip_install(packages)

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if DEVICE == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if not torch.cuda.is_available():
    raise RuntimeError("StableCodec needs CUDA for practical runtime. Enable a Kaggle GPU accelerator.")


In [ ]:
def find_existing_repo() -> Path | None:
    roots = [Path("/kaggle/input"), Path.cwd(), WORK]
    for root in roots:
        if not root.exists():
            continue
        for hit in root.rglob("compress.sh"):
            repo = hit.parent
            if (repo / "src" / "StableCodec.py").exists() and (repo / "src" / "compress.py").exists():
                return repo
    return None

if not (REPO_DIR / "src" / "StableCodec.py").exists():
    existing = find_existing_repo()
    if existing is not None and existing.resolve() != REPO_DIR.resolve():
        print("Copying StableCodec repo from", existing)
        shutil.copytree(existing, REPO_DIR, dirs_exist_ok=True)
    else:
        run(["git", "clone", "--depth", "1", STABLECODEC_REPO_URL, REPO_DIR])
else:
    print("Using StableCodec repo at", REPO_DIR)

def replace_text(path: Path, old: str, new: str) -> None:
    text = path.read_text(encoding="utf-8")
    if old in text:
        path.write_text(text.replace(old, new), encoding="utf-8")
        print("patched", path.relative_to(REPO_DIR))

# Make color_fix opt-in if this repo has the older default.
testing = REPO_DIR / "src" / "my_utils" / "testing_utils.py"
if testing.exists():
    replace_text(
        testing,
        'parser.add_argument("--color_fix", default=True, help="Whether or not to use color fix.")',
        'parser.add_argument("--color_fix", action="store_true", help="Whether or not to use source-image color fix.")',
    )

print("StableCodec ready:", REPO_DIR)


In [ ]:
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def expected_min_bytes(filename: str) -> int:
    return EXPECTED_MIN_BYTES.get(filename, 1)

def find_existing_file(filename: str, min_bytes: int | None = None) -> Path | None:
    if min_bytes is None:
        min_bytes = expected_min_bytes(filename)
    direct = STABLECODEC_CHECKPOINT_INPUT_DIR / filename
    if direct.exists() and direct.is_file() and direct.stat().st_size >= min_bytes:
        print("Using explicit checkpoint path", direct)
        return direct
    for root in [STABLECODEC_CHECKPOINT_INPUT_DIR, CKPT_DIR, MODEL_DIR, KAGGLE_INPUT, Path.cwd(), WORK]:
        if not root.exists():
            continue
        for p in sorted(root.rglob(filename)):
            if p.is_file() and p.stat().st_size >= min_bytes:
                return p
    return None

def ensure_gdrive_file(filename: str) -> Path:
    existing = find_existing_file(filename)
    if existing is not None:
        print(f"Found {filename} -> {existing}")
        return existing
    if filename not in GDRIVE_IDS:
        raise FileNotFoundError(f"No known Google Drive id for {filename}")
    import gdown
    out = CKPT_DIR / filename
    print(f"Downloading {filename}...")
    gdown.download(
        url=f"https://drive.google.com/uc?id={GDRIVE_IDS[filename]}",
        output=str(out),
        quiet=False,
        use_cookies=False,
    )
    if not out.exists() or out.stat().st_size < expected_min_bytes(filename):
        raise RuntimeError(
            f"Failed to download {filename}. Upload it as a Kaggle dataset if Google Drive quota blocks the download."
        )
    return out

elic_path = ensure_gdrive_file("elic_official.pth")
ft8_path = ensure_gdrive_file("stablecodec_ft8.pkl")

from huggingface_hub import snapshot_download

def find_sd_turbo() -> Path | None:
    for root in [MODEL_DIR, KAGGLE_INPUT, Path.cwd(), WORK]:
        if not root.exists():
            continue
        for p in root.rglob("model_index.json"):
            if p.parent.name == "sd-turbo" or (p.parent / "unet").exists():
                return p.parent
    return None

existing_sd = find_sd_turbo()
if existing_sd is not None:
    sd_path = existing_sd
    print("Using SD-Turbo at", sd_path)
else:
    print("Downloading SD-Turbo from Hugging Face...")
    sd_path = Path(snapshot_download(
        repo_id="stabilityai/sd-turbo",
        local_dir=str(SD_TURBO_DIR),
        local_dir_use_symlinks=False,
    ))

print("ft8:", ft8_path)
print("ELIC:", elic_path)
print("SD_TURBO:", sd_path)


In [ ]:
@contextmanager
def pushd(path: Path):
    old = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)

sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

with pushd(REPO_DIR / "src"):
    from accelerate.utils import set_seed
    from diffusers.utils.import_utils import is_xformers_available
    from StableCodec import StableCodec
    from my_utils.compress_utils import write_body, read_body

if SEED is not None:
    set_seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

sc_args = SimpleNamespace(
    sd_path=str(sd_path),
    elic_path=str(elic_path),
    codec_path=str(ft8_path),
    img_path="",
    rec_path="",
    bin_path="",
    lora_rank_unet=32,
    lora_rank_vae=16,
    vae_decoder_tiled_size=160,
    vae_encoder_tiled_size=1024,
    latent_tiled_size=96,
    latent_tiled_overlap=32,
    pos_prompt="A high-resolution, 8K, ultra-realistic image with sharp focus, vibrant colors, and natural lighting.",
    seed=SEED,
    enable_xformers_memory_efficient_attention=ENABLE_XFORMERS,
    set_grads_to_none=False,
    world_size=1,
    local_rank=-1,
    dist_url="env://",
    lambda_rate=0.5,
    color_fix=False,
)

with pushd(REPO_DIR / "src"):
    net = StableCodec(sd_path=str(sd_path), args=sc_args)
net.cuda().eval()
net.codec.update(force=True)
if ENABLE_XFORMERS:
    if is_xformers_available():
        net.unet.enable_xformers_memory_efficient_attention()
    else:
        raise RuntimeError("xformers requested but not available")
print("StableCodec ft8 loaded")


In [ ]:
def image_to_tensor(image: Image.Image) -> torch.Tensor:
    arr = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return (tensor * 2.0 - 1.0).cuda()

def tensor_to_pil(tensor: torch.Tensor) -> Image.Image:
    from torchvision import transforms
    return transforms.ToPILImage()(tensor[0].clamp(0.0, 1.0))

def compress_decode_one(image_lr: Image.Image, raw_bitstream_path: Path) -> tuple[Image.Image, int, float, float]:
    lr_w, lr_h = image_lr.size
    x = image_to_tensor(image_lr)
    pad_h = math.ceil(lr_h / 256) * 256 - lr_h
    pad_w = math.ceil(lr_w / 256) * 256 - lr_w
    pad_mode = "reflect" if pad_h < lr_h and pad_w < lr_w else "replicate"
    x_padded = torch.nn.functional.pad(x, pad=(0, pad_w, 0, pad_h), mode=pad_mode)

    raw_bitstream_path.parent.mkdir(parents=True, exist_ok=True)
    with torch.no_grad():
        t0 = time.perf_counter()
        output_dict = net.compress(x_padded)
        with raw_bitstream_path.open("wb") as f:
            write_body(f, output_dict["shape"], output_dict["strings"])
        encode_time = time.perf_counter() - t0

        t1 = time.perf_counter()
        with raw_bitstream_path.open("rb") as f:
            strings, shape = read_body(f)
        out_img = net.decompress(strings, shape, [1])
        out_img = out_img[:, :, 0:lr_h, 0:lr_w]
        out_img = (out_img * 0.5 + 0.5).float().cpu().detach()
        decode_time = time.perf_counter() - t1

    return tensor_to_pil(out_img), raw_bitstream_path.stat().st_size, encode_time, decode_time

def header_bytes(scale: float, downsampler: str, upscaler: str, orig_h: int, orig_w: int, lr_h: int, lr_w: int) -> bytes:
    return struct.pack(
        HEADER_STRUCT,
        HEADER_MAGIC,
        HEADER_VERSION,
        SCALE_IDS[float(scale)],
        METHOD_IDS[downsampler],
        METHOD_IDS[upscaler],
        int(orig_h),
        int(orig_w),
        int(lr_h),
        int(lr_w),
    )

def parse_header(payload: bytes) -> dict:
    magic, version, scale_id, down_id, up_id, orig_h, orig_w, lr_h, lr_w = struct.unpack(
        HEADER_STRUCT, payload[:HEADER_LEN]
    )
    if magic != HEADER_MAGIC or version != HEADER_VERSION:
        raise ValueError("bad SCDS1 header")
    return {
        "scale_id": scale_id,
        "scale": SCALES[scale_id],
        "downsampler": ID_TO_METHOD[down_id],
        "upscaler": ID_TO_METHOD[up_id],
        "orig_h": orig_h,
        "orig_w": orig_w,
        "lr_h": lr_h,
        "lr_w": lr_w,
    }


In [ ]:
import torch.nn.functional as F
import torchvision.transforms.functional as TF

METRIC_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _fspecial(size=11, sigma=1.5):
    mm = nn_ = (size - 1) / 2.0
    y, x = np.ogrid[-mm:mm + 1, -nn_:nn_ + 1]
    h = np.exp(-(x * x + y * y) / (2.0 * sigma * sigma))
    h[h < np.finfo(h.dtype).eps * h.max()] = 0
    sm = h.sum()
    if sm != 0:
        h /= sm
    return torch.from_numpy(h)

def _to_y255(x):
    w = torch.tensor([0.299, 0.587, 0.114], dtype=x.dtype, device=x.device).view(1, 3, 1, 1)
    y = (x * w).sum(1, keepdim=True) * 255.0
    return (y - y.detach() + y.round()).to(torch.float32)

def _ssim(X, Y, win, dr=255.0):
    C1, C2 = (0.01 * dr) ** 2, (0.03 * dr) ** 2
    mu1, mu2 = F.conv2d(X, win, padding=0), F.conv2d(Y, win, padding=0)
    m1s, m2s, m12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1 = F.conv2d(X * X, win, padding=0) - m1s
    s2 = F.conv2d(Y * Y, win, padding=0) - m2s
    s12 = F.conv2d(X * Y, win, padding=0) - m12
    cs = F.relu((2 * s12 + C2) / (s1 + s2 + C2))
    ssim_map = ((2 * m12 + C1) / (m1s + m2s + C1)) * cs
    return ssim_map.mean([1, 2, 3]), cs.mean([1, 2, 3])

def ms_ssim_score(rec01, ref01):
    X, Y = _to_y255(rec01), _to_y255(ref01)
    win = _fspecial().to(torch.float32).view(1, 1, 11, 11).to(X.device)
    wts = torch.tensor([0.0448, 0.2856, 0.3001, 0.2363, 0.1333], dtype=torch.float32, device=X.device)
    mcs, sv = [], None
    for _ in range(5):
        sv, cs = _ssim(X, Y, win)
        mcs.append(cs)
        pad = (X.shape[2] % 2, X.shape[3] % 2)
        X, Y = F.avg_pool2d(X, 2, padding=pad), F.avg_pool2d(Y, 2, padding=pad)
    mcs = torch.stack(mcs, 0)
    return (torch.prod(mcs[:-1] ** wts[:-1].unsqueeze(1), 0) * (sv ** wts[-1])).item()

def psnr_score(rec01, ref01):
    mse = torch.mean((rec01 - ref01) ** 2)
    return (10 * torch.log10(1.0 / mse)).item() if mse > 0 else 99.0

_LP = _DS = None
def metric_backbones():
    global _LP, _DS
    if _LP is None:
        import lpips as _lpips
        import DISTS_pytorch
        from DISTS_pytorch import DISTS
        print("Loading LPIPS(net='alex') + DISTS(weights.pt)")
        _LP = _lpips.LPIPS(net="alex", verbose=False).to(METRIC_DEVICE).eval()
        ds = DISTS(load_weights=False)
        wp = os.path.join(os.path.dirname(DISTS_pytorch.__file__), "weights.pt")
        weights = torch.load(wp, map_location="cpu")
        ds.alpha.data, ds.beta.data = weights["alpha"], weights["beta"]
        _DS = ds.to(METRIC_DEVICE).eval()
    return _LP, _DS

def image_tensor(path: Path):
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    return TF.to_tensor(img).unsqueeze(0).to(METRIC_DEVICE)

def score_pair(rec_path: Path, src_path: Path) -> dict:
    rec = image_tensor(rec_path)
    ref = image_tensor(src_path)
    if rec.shape != ref.shape:
        raise ValueError(f"resolution mismatch {rec_path.name}: rec={tuple(rec.shape[2:])} src={tuple(ref.shape[2:])}")
    lp, ds = metric_backbones()
    with torch.inference_mode():
        psnr = psnr_score(rec, ref)
        ms = ms_ssim_score(rec, ref)
        lpips_v = lp(rec * 2 - 1, ref * 2 - 1).item()
        dists_v = ds(rec, ref).item()
    final = psnr + 10.0 * ms + 20.0 * (1.0 - lpips_v) + 25.0 * (1.0 - dists_v)
    return {"PSNR": psnr, "MS_SSIM": ms, "LPIPS": lpips_v, "DISTS": dists_v, "Final_Score": final}


In [ ]:
reset_dir(OUT_ROOT)
all_rows = []
options_by_image: dict[str, list[dict]] = {p.name: [] for p in input_paths}
setting_summaries = []

for scale in SCALES:
    if not (0 < float(scale) <= 1.0):
        raise ValueError(f"Invalid scale: {scale}")
    for downsampler in DOWNSAMPLERS:
        name = setting_name(float(scale), downsampler)
        root = OUT_ROOT / name
        down_dir = root / "downsampled"
        raw_bin_dir = root / "raw_bitstream"
        decoded_lr_dir = root / "decoded_lr"
        final_dir = root / "final_hr"
        for d in [down_dir, raw_bin_dir, decoded_lr_dir, final_dir]:
            d.mkdir(parents=True, exist_ok=True)

        print(f"\n=== Running setting {name} ===")
        rows_for_setting = []
        start_setting = time.perf_counter()

        for idx, src in enumerate(input_paths, start=1):
            total_start = time.perf_counter()
            original = open_rgb(src)
            orig_w, orig_h = original.size
            lr_w, lr_h = round_scaled_size(orig_w, orig_h, float(scale))
            downsampled = original.resize((lr_w, lr_h), pil_resample(downsampler))
            down_path = down_dir / src.name
            if SAVE_CANDIDATE_IMAGES:
                downsampled.save(down_path)

            raw_bit = raw_bin_dir / src.stem
            decoded_lr, raw_bytes, encode_time, decode_time = compress_decode_one(downsampled, raw_bit)
            if SAVE_CANDIDATE_IMAGES:
                decoded_lr.save(decoded_lr_dir / src.name)

            for upscaler in UPSCALERS:
                up_start = time.perf_counter()
                final = decoded_lr.resize((orig_w, orig_h), pil_resample(upscaler))
                upscale_time = time.perf_counter() - up_start
                stem = candidate_stem(src.stem, float(scale), downsampler, upscaler)
                final_path = final_dir / f"{stem}_final_hr.png"
                final.save(final_path)

                headered_bytes = raw_bytes + HEADER_LEN
                actual_bits = headered_bytes * 8
                nominal_lr_bpp = actual_bits / (lr_w * lr_h)
                effective_bpp = actual_bits / (orig_w * orig_h)
                metrics = score_pair(final_path, src) if RUN_FULL_METRIC_SELECTION else {
                    "PSNR": None, "MS_SSIM": None, "LPIPS": None, "DISTS": None,
                    "Final_Score": -float(headered_bytes),
                }
                total_time = time.perf_counter() - total_start
                row = {
                    "image_name": src.name,
                    "checkpoint_name": "stablecodec_ft8.pkl",
                    "original_height": orig_h,
                    "original_width": orig_w,
                    "scale": float(scale),
                    "downsampled_height": lr_h,
                    "downsampled_width": lr_w,
                    "downsampler": downsampler,
                    "upscaler": upscaler,
                    "bitstream_bytes": headered_bytes,
                    "actual_bits": actual_bits,
                    "nominal_lr_bpp": nominal_lr_bpp,
                    "effective_bpp": effective_bpp,
                    "bpp_budget_pass": effective_bpp <= BPP_LIMIT,
                    **metrics,
                    "encode_time_sec": encode_time,
                    "decode_time_sec": decode_time,
                    "upscale_time_sec": upscale_time,
                    "total_time_sec": total_time,
                    "raw_bitstream_path": str(raw_bit),
                    "final_path": str(final_path),
                    "scale_id": SCALE_IDS[float(scale)],
                    "down_id": METHOD_IDS[downsampler],
                    "up_id": METHOD_IDS[upscaler],
                    "raw_bytes": raw_bytes,
                }
                all_rows.append(row)
                rows_for_setting.append(row)
                options_by_image[src.name].append(row)

            if idx % 5 == 0 or idx == len(input_paths):
                print(f"  {name}: processed {idx}/{len(input_paths)}")
            torch.cuda.empty_cache()

        setting_elapsed = time.perf_counter() - start_setting
        avg_eff = sum(float(r["effective_bpp"]) for r in rows_for_setting) / len(rows_for_setting)
        avg_score = None
        valid_scores = [float(r["Final_Score"]) for r in rows_for_setting if r.get("Final_Score") is not None]
        if valid_scores:
            avg_score = sum(valid_scores) / len(valid_scores)
        setting_summaries.append({
            "setting": name,
            "scale": float(scale),
            "downsampler": downsampler,
            "candidate_rows": len(rows_for_setting),
            "average_effective_bpp": avg_eff,
            "average_Final_Score": avg_score,
            "runtime_sec": setting_elapsed,
        })
        write_csv(OUT_ROOT / "all_results.csv", all_rows, CSV_COLUMNS)

print("Generated candidate rows:", len(all_rows))
(OUT_ROOT / "setting_summaries.json").write_text(json.dumps(setting_summaries, indent=2), encoding="utf-8")
write_csv(OUT_ROOT / "all_results.csv", all_rows, CSV_COLUMNS)
write_csv(OUT_ROOT / "budget_valid_results.csv", [r for r in all_rows if r["effective_bpp"] <= BPP_LIMIT], CSV_COLUMNS)


In [ ]:
# Multiple-choice knapsack: one selected candidate per image under the global average-bpp budget.
image_names = [p.name for p in input_paths]
ordered_options = []
for name in image_names:
    opts = options_by_image[name]
    if not opts:
        raise RuntimeError(f"No candidate options for {name}")
    opts = sorted(opts, key=lambda r: (int(r["bitstream_bytes"]), -float(r["Final_Score"])))
    ordered_options.append(opts)

cap = byte_budget
n = len(ordered_options)
dp = np.full(cap + 1, -1e30, dtype=np.float64)
dp[0] = 0.0
choice = np.full((n, cap + 1), -1, dtype=np.int16)

for i, opts in enumerate(ordered_options):
    new = np.full(cap + 1, -1e30, dtype=np.float64)
    new_choice = np.full(cap + 1, -1, dtype=np.int16)
    for j, opt in enumerate(opts):
        cost = int(opt["bitstream_bytes"])
        value = float(opt["Final_Score"])
        if cost > cap:
            continue
        cand_values = dp[: cap + 1 - cost] + value
        target = new[cost:]
        mask = cand_values > target
        target[mask] = cand_values[mask]
        new_choice[cost:][mask] = j
    dp = new
    choice[i] = new_choice
    if np.max(dp) < -1e20:
        raise RuntimeError(f"Knapsack failed at image {image_names[i]}; byte budget too low.")

best_bytes = int(np.argmax(dp))
best_total_score = float(dp[best_bytes])
selected_options = []
b = best_bytes
for i in range(n - 1, -1, -1):
    j = int(choice[i, b])
    if j < 0:
        raise RuntimeError(f"Could not reconstruct DP choice at image index {i}, bytes={b}")
    opt = ordered_options[i][j]
    selected_options.append(opt)
    b -= int(opt["bitstream_bytes"])
selected_options.reverse()

if b != 0:
    raise RuntimeError("Internal DP reconstruction error")

selected_bytes = sum(int(r["bitstream_bytes"]) for r in selected_options)
selected_bpp = 8.0 * selected_bytes / total_pixels
print("Selected bytes:", selected_bytes, "/", absolute_byte_budget)
print("Selected bpp:", selected_bpp)
print("Selected average score:", best_total_score / n)
if selected_bpp > BPP_LIMIT:
    raise RuntimeError(f"Selected BPP too high: {selected_bpp:.6f} > {BPP_LIMIT}")

counts = {}
for r in selected_options:
    key = f"s{scale_tag(float(r['scale']))}_{r['downsampler']}_{r['upscaler']}"
    counts[key] = counts.get(key, 0) + 1
print("Selection counts:", counts)

# Requested analysis CSVs.
best_per_image = []
for name in image_names:
    valid = [r for r in options_by_image[name] if r["effective_bpp"] <= BPP_LIMIT and r.get("Final_Score") is not None]
    if valid:
        best_per_image.append(max(valid, key=lambda r: float(r["Final_Score"])))
write_csv(OUT_ROOT / "best_per_image.csv", best_per_image, CSV_COLUMNS)

grouped = {}
for row in all_rows:
    if row.get("Final_Score") is None or row["effective_bpp"] > BPP_LIMIT:
        continue
    key = (row["scale"], row["downsampler"], row["upscaler"])
    grouped.setdefault(key, []).append(row)

summary_rows = []
for (scale, downsampler, upscaler), rows in grouped.items():
    def avg(field):
        vals = [float(r[field]) for r in rows if r.get(field) is not None]
        return sum(vals) / len(vals) if vals else None
    summary_rows.append({
        "checkpoint": "stablecodec_ft8.pkl",
        "scale": scale,
        "downsampler": downsampler,
        "upscaler": upscaler,
        "image_count": len(rows),
        "best_average_Final_Score": avg("Final_Score"),
        "average_PSNR": avg("PSNR"),
        "average_MS_SSIM": avg("MS_SSIM"),
        "average_LPIPS": avg("LPIPS"),
        "average_DISTS": avg("DISTS"),
        "average_effective_bpp": avg("effective_bpp"),
    })
summary_rows.sort(key=lambda r: float(r["best_average_Final_Score"] or -1e30), reverse=True)
summary_cols = [
    "checkpoint", "scale", "downsampler", "upscaler", "image_count",
    "best_average_Final_Score", "average_PSNR", "average_MS_SSIM",
    "average_LPIPS", "average_DISTS", "average_effective_bpp",
]
write_csv(OUT_ROOT / "summary_by_setting.csv", summary_rows, summary_cols)
write_csv(OUT_ROOT / "best_overall_summary.csv", summary_rows[:1], summary_cols)

selected_manifest = []
for row in selected_options:
    selected_manifest.append({
        k: row[k] for k in [
            "image_name", "scale", "downsampled_height", "downsampled_width",
            "downsampler", "upscaler", "bitstream_bytes", "raw_bytes",
            "effective_bpp", "nominal_lr_bpp", "PSNR", "MS_SSIM",
            "LPIPS", "DISTS", "Final_Score",
        ]
    })
(OUT_ROOT / "selected_manifest.json").write_text(json.dumps(selected_manifest, indent=2), encoding="utf-8")


In [ ]:
reset_dir(FINAL_ROOT)
rec_out = FINAL_ROOT / "reconstructed"
bit_out = FINAL_ROOT / "bitstream"
rec_out.mkdir(parents=True, exist_ok=True)
bit_out.mkdir(parents=True, exist_ok=True)

for row in selected_options:
    src_name = row["image_name"]
    final_path = Path(row["final_path"])
    raw_path = Path(row["raw_bitstream_path"])
    if not final_path.exists():
        raise FileNotFoundError(f"Missing selected reconstruction: {final_path}")
    if not raw_path.exists():
        raise FileNotFoundError(f"Missing selected raw bitstream: {raw_path}")

    shutil.copy2(final_path, rec_out / src_name)
    header = header_bytes(
        float(row["scale"]),
        row["downsampler"],
        row["upscaler"],
        int(row["original_height"]),
        int(row["original_width"]),
        int(row["downsampled_height"]),
        int(row["downsampled_width"]),
    )
    payload = header + raw_path.read_bytes()
    out_bin = bit_out / f"{Path(src_name).stem}.bin"
    out_bin.write_bytes(payload)
    # Quick self-check of counted metadata.
    parsed = parse_header(payload)
    assert parsed["orig_h"] == int(row["original_height"])
    assert parsed["orig_w"] == int(row["original_width"])
    assert parsed["lr_h"] == int(row["downsampled_height"])
    assert parsed["lr_w"] == int(row["downsampled_width"])

def validate_submission(root: Path) -> dict:
    total_bytes = 0
    errors = []
    for src in input_paths:
        rec = root / "reconstructed" / src.name
        bit = root / "bitstream" / f"{src.stem}.bin"
        if not rec.exists():
            errors.append(f"missing rec {src.name}")
            continue
        if not bit.exists():
            errors.append(f"missing bit {src.stem}.bin")
            continue
        payload = bit.read_bytes()
        if len(payload) <= HEADER_LEN:
            errors.append(f"too small bitstream {src.stem}.bin")
        else:
            try:
                parse_header(payload)
            except Exception as exc:
                errors.append(f"bad header {src.stem}.bin: {exc}")
        with Image.open(src) as gt, Image.open(rec) as rr:
            if rr.size != gt.size:
                errors.append(f"size mismatch {src.name}: rec={rr.size}, src={gt.size}")
        total_bytes += bit.stat().st_size
    avg_bpp = 8.0 * total_bytes / total_pixels
    return {"errors": errors, "bytes": total_bytes, "avg_bpp": avg_bpp}

validation = validate_submission(FINAL_ROOT)
print("Validation errors:", len(validation["errors"]))
print("Final avg_bpp:", validation["avg_bpp"])
if validation["errors"]:
    print("\n".join(validation["errors"][:20]))
    raise RuntimeError("Invalid wrapped submission")
if validation["avg_bpp"] > BPP_LIMIT:
    raise RuntimeError(f"BPP too high: {validation['avg_bpp']:.6f} > {BPP_LIMIT}")

counts_text = ", ".join(f"{k}:{v}" for k, v in sorted(counts.items()))
avg_runtime = sum(float(r["total_time_sec"]) for r in selected_options) / max(1, len(selected_options))
readme = (
    f"runtime per image [s] : {avg_runtime:.4f}\n"
    "CPU[1] / GPU[0] : 0\n"
    "Extra Data [1] / No Extra Data [0] : 1\n"
    "Other description: No-training pretrained StableCodec ft8 downsample/upscale submission. "
    "Only stablecodec_ft8.pkl is used. No ft16/ft24/ft32. "
    f"Scales tested: {SCALES}; downsamplers: {DOWNSAMPLERS}; upscalers: {UPSCALERS}. "
    f"Selected counts: {counts_text}. "
    "No source-image color fix or post-decode original-image statistics were used. "
    "Each .bin contains a counted SCDS1 metadata header plus a real StableCodec ft8 entropy stream; "
    "the header records scale, resize methods, original size, and decoded low-resolution size for a compatible decoder. "
    f"Average BPP measured from submitted .bin files: {validation['avg_bpp']:.6f}.\n"
)
(FINAL_ROOT / "readme.txt").write_text(readme, encoding="utf-8")
(FINAL_ROOT / "selected_manifest.json").write_text(json.dumps(selected_manifest, indent=2), encoding="utf-8")

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted((FINAL_ROOT / "reconstructed").glob("*.png")):
        zf.write(p, f"reconstructed/{p.name}")
    for p in sorted((FINAL_ROOT / "bitstream").glob("*.bin")):
        zf.write(p, f"bitstream/{p.name}")
    zf.write(FINAL_ROOT / "readme.txt", "readme.txt")

print("Created submission:", FINAL_ZIP)
print(readme)


In [ ]:
if RUN_STRUCTURE_EVALUATOR:
    run([
        sys.executable,
        "scripts/evaluate_submission.py",
        INPUT_DIR,
        FINAL_ZIP,
        "--structure-only",
        "--json",
        WORK / "ft8_downscale_structure_results.json",
        "--csv",
        WORK / "ft8_downscale_structure_per_image.csv",
    ], check=True)
else:
    print("Structure evaluator skipped.")


In [ ]:
SCORE_SCRIPT_SOURCE = '#!/usr/bin/env python3\n"""\nBoard-faithful local scorer for a LoViF submission ZIP (or folder).\n\nScoring strategy is ported VERBATIM from cod-lite-stage1.ipynb, which is confirmed\nto match the Codabench leaderboard:\n  - LPIPS : the `lpips` package, LPIPS(net=\'alex\'), REAL pretrained weights   (NOT pyiqa, NOT pretrained=False)\n  - DISTS : DISTS_pytorch with its shipped weights.pt                         (REAL pretrained)\n  - PSNR  : hand-written, matches pyiqa\n  - MS-SSIM: hand-written on the Y channel, matches pyiqa to <1e-5\n  Final = PSNR + 10*MS_SSIM + 20*(1-LPIPS) + 25*(1-DISTS)\n\nIt scores the reconstructed/*.png in the ZIP against dataset_val/*.png and ALSO\nreports avg_bpp from bitstream/*.bin (budget 0.008).\n\nUSAGE ? just edit the CONFIG block below, then:  python score_zip_boardfaithful.py\n(No CLI args ? paths with spaces broke argparse before.)\n"""\n\nimport io\nimport os\nimport sys\nimport time\nimport zipfile\nimport tempfile\nimport subprocess\nfrom pathlib import Path\n\n# ============================== CONFIG (edit these) ==============================\nZIP_PATH   = "C:\\\\Users\\\\Xiaomi\\\\Desktop\\\\eccv\\\\lovif\\\\lovif_stablecodec_submission2.zip"          # zip OR a folder containing reconstructed/ + bitstream/\nGT_DIR     = "C:\\\\Users\\\\Xiaomi\\\\Desktop\\\\eccv\\\\lovif\\\\filtered_dataset\\\\dataset_val"         # ground-truth originals (100 pngs)\nLIMIT      = None                     # int = score only the first N images (e.g. 10 for a quick check); None = all\nDEVICE     = "auto"                    # "auto" | "cpu" | "cuda" | "mps"\nHISTORY    = "score_history_boardfaithful.txt"   # appends each run here\nBUDGET_BPP = 0.008\n# ================================================================================\n\nimport numpy as np\nfrom PIL import Image\nImage.MAX_IMAGE_PIXELS = None\n\n\ndef log(msg):\n    print(msg, flush=True)\n\n\ndef ensure_deps():\n    need = []\n    for mod, pip_name in [("torch", "torch"), ("torchvision", "torchvision"),\n                          ("lpips", "lpips"), ("DISTS_pytorch", "DISTS_pytorch")]:\n        try:\n            __import__(mod)\n        except Exception:\n            need.append(pip_name)\n    if need:\n        log(f"installing missing deps: {need}")\n        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=False)\n\n\nensure_deps()\nimport torch\nimport torch.nn.functional as F\nimport torchvision.transforms.functional as TF\n\n\ndef pick_device(want):\n    if want != "auto":\n        return want\n    if torch.cuda.is_available():\n        return "cuda"\n    # NOTE: MPS breaks pyiqa-style float64; our scorer is float32, but MS-SSIM avg_pool on\n    # tiny tensors is fine on MPS. Still default to cpu for exact reproducibility.\n    return "cpu"\n\n\nDEV = pick_device(DEVICE)\nlog(f"device: {DEV}")\n\n\n# ----------------------- metric implementations (cod-lite cell 13) -----------------------\ndef _fspecial(size=11, sigma=1.5):\n    mm = nn_ = (size - 1) / 2.0\n    y, x = np.ogrid[-mm:mm + 1, -nn_:nn_ + 1]\n    h = np.exp(-(x * x + y * y) / (2.0 * sigma * sigma))\n    h[h < np.finfo(h.dtype).eps * h.max()] = 0\n    sm = h.sum()\n    if sm != 0:\n        h /= sm\n    return torch.from_numpy(h)\n\n\ndef _to_y255(x):\n    w = torch.tensor([0.299, 0.587, 0.114], dtype=x.dtype, device=x.device).view(1, 3, 1, 1)\n    y = (x * w).sum(1, keepdim=True) * 255.0\n    return (y - y.detach() + y.round()).to(torch.float32)\n\n\ndef _ssim(X, Y, win, dr=255.0):\n    C1, C2 = (0.01 * dr) ** 2, (0.03 * dr) ** 2\n    mu1, mu2 = F.conv2d(X, win, padding=0), F.conv2d(Y, win, padding=0)\n    m1s, m2s, m12 = mu1 ** 2, mu2 ** 2, mu1 * mu2\n    s1 = F.conv2d(X * X, win, padding=0) - m1s\n    s2 = F.conv2d(Y * Y, win, padding=0) - m2s\n    s12 = F.conv2d(X * Y, win, padding=0) - m12\n    cs = F.relu((2 * s12 + C2) / (s1 + s2 + C2))\n    ssim_map = ((2 * m12 + C1) / (m1s + m2s + C1)) * cs\n    return ssim_map.mean([1, 2, 3]), cs.mean([1, 2, 3])\n\n\ndef ms_ssim_score(rec01, ref01):\n    X, Y = _to_y255(rec01), _to_y255(ref01)\n    win = _fspecial().to(torch.float32).view(1, 1, 11, 11).to(X.device)\n    wts = torch.tensor([0.0448, 0.2856, 0.3001, 0.2363, 0.1333], dtype=torch.float32, device=X.device)\n    mcs, sv = [], None\n    for _ in range(5):\n        sv, cs = _ssim(X, Y, win)\n        mcs.append(cs)\n        pad = (X.shape[2] % 2, X.shape[3] % 2)\n        X, Y = F.avg_pool2d(X, 2, padding=pad), F.avg_pool2d(Y, 2, padding=pad)\n    mcs = torch.stack(mcs, 0)\n    return (torch.prod(mcs[:-1] ** wts[:-1].unsqueeze(1), 0) * (sv ** wts[-1])).item()\n\n\ndef psnr_score(rec01, ref01):\n    return (10 * torch.log10(1.0 / torch.mean((rec01 - ref01) ** 2))).item()\n\n\n_LP = _DS = None\ndef _scorers():\n    global _LP, _DS\n    if _LP is None:\n        import lpips as _l\n        import DISTS_pytorch\n        from DISTS_pytorch import DISTS as _D\n        log("loading LPIPS(net=\'alex\') + DISTS(weights.pt) ...")\n        _LP = _l.LPIPS(net=\'alex\', verbose=False).to(DEV).eval()\n        ds = _D(load_weights=False)\n        wp = os.path.join(os.path.dirname(DISTS_pytorch.__file__), \'weights.pt\')\n        w = torch.load(wp, map_location=\'cpu\')\n        ds.alpha.data, ds.beta.data = w[\'alpha\'], w[\'beta\']\n        _DS = ds.to(DEV).eval()\n        log("metric backbones ready.")\n    return _LP, _DS\n\n\ndef score_pair(rec_path, ref_path):\n    rec = TF.to_tensor(Image.open(rec_path).convert(\'RGB\')).unsqueeze(0).to(DEV)\n    ref = TF.to_tensor(Image.open(ref_path).convert(\'RGB\')).unsqueeze(0).to(DEV)\n    if rec.shape != ref.shape:\n        raise ValueError(f"resolution mismatch rec{tuple(rec.shape[2:])} vs ref{tuple(ref.shape[2:])}")\n    lp, ds = _scorers()\n    with torch.inference_mode():\n        return {\'psnr\': psnr_score(rec, ref),\n                \'ms_ssim\': ms_ssim_score(rec, ref),\n                \'lpips\': lp(rec * 2 - 1, ref * 2 - 1).item(),\n                \'dists\': ds(rec, ref).item()}\n\n\n# ----------------------------- submission handling -----------------------------\ndef prepare(path, tmp):\n    p = Path(path)\n    if p.is_dir():\n        return p\n    if p.is_file() and p.suffix.lower() == ".zip":\n        out = Path(tmp) / "sub"\n        out.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(p) as z:\n            z.extractall(out)\n        # the zip may wrap everything in a top folder; find the dir holding reconstructed/\n        if (out / "reconstructed").is_dir():\n            return out\n        for sub in out.rglob("reconstructed"):\n            if sub.is_dir():\n                return sub.parent\n        return out\n    raise FileNotFoundError(f"not a zip or folder: {path}")\n\n\ndef main():\n    t_start = time.time()\n    gt = sorted(Path(GT_DIR).glob("*.png"))\n    if not gt:\n        log(f"ERROR: no PNGs in GT_DIR={GT_DIR}")\n        return 1\n    if LIMIT:\n        gt = gt[:LIMIT]\n    log(f"ground truth: {len(gt)} images from {GT_DIR}" + (f"  (LIMIT={LIMIT})" if LIMIT else ""))\n\n    with tempfile.TemporaryDirectory() as tmp:\n        sub = prepare(ZIP_PATH, tmp)\n        rec_dir = sub / "reconstructed"\n        bin_dir = sub / "bitstream"\n        log(f"submission: {sub}")\n        if not rec_dir.is_dir():\n            log(f"ERROR: missing {rec_dir}")\n            return 1\n\n        rows = []\n        tot_bytes = tot_px = 0\n        acc = {k: 0.0 for k in ("psnr", "ms_ssim", "lpips", "dists")}\n        n = len(gt)\n        log("\\nscoring:")\n        for i, gp in enumerate(gt, 1):\n            rp = rec_dir / gp.name\n            if not rp.exists():\n                log(f"  [{i:03d}/{n}] {gp.name}  MISSING reconstructed -> skipped")\n                continue\n            with Image.open(gp) as oi:\n                w, h = oi.size\n            px = w * h\n            sc = score_pair(rp, gp)\n            for k in acc:\n                acc[k] += sc[k]\n            bp = bin_dir / f"{gp.stem}.bin"\n            bbytes = bp.stat().st_size if bp.exists() else 0\n            tot_bytes += bbytes\n            tot_px += px\n            bpp = bbytes * 8 / px if px else 0\n            rows.append((gp.name, sc, bpp))\n            log(f"  [{i:03d}/{n}] {gp.name:<14} PSNR {sc[\'psnr\']:6.3f} | MS-SSIM {sc[\'ms_ssim\']:.4f} | "\n                f"LPIPS {sc[\'lpips\']:.4f} | DISTS {sc[\'dists\']:.4f} | bpp {bpp:.5f} | "\n                f"{(time.time()-t_start)/i:.2f}s/img")\n\n        m = len(rows)\n        if m == 0:\n            log("ERROR: no scored images.")\n            return 1\n        means = {k: acc[k] / m for k in acc}\n        final = (means[\'psnr\'] + 10.0 * means[\'ms_ssim\']\n                 + 20.0 * (1.0 - means[\'lpips\']) + 25.0 * (1.0 - means[\'dists\']))\n        avg_bpp = tot_bytes * 8 / tot_px if tot_px else 0.0\n        max_bpp = max((r[2] for r in rows), default=0.0)\n        within = avg_bpp <= BUDGET_BPP\n\n        log("\\n" + "=" * 70)\n        log(f"  BOARD-FAITHFUL SCORE  ({m} images)   zip={Path(ZIP_PATH).name}")\n        log("-" * 70)\n        log(f"  PSNR        : {means[\'psnr\']:.4f}")\n        log(f"  MS-SSIM     : {means[\'ms_ssim\']:.4f}")\n        log(f"  LPIPS       : {means[\'lpips\']:.4f}   (lower better)")\n        log(f"  DISTS       : {means[\'dists\']:.4f}   (lower better)")\n        log(f"  avg_bpp     : {avg_bpp:.6f}  /  {BUDGET_BPP}   ({\'OK\' if within else \'OVER BUDGET!\'})")\n        log(f"  max_img_bpp : {max_bpp:.6f}")\n        log(f"  >>> FINAL_SCORE = {final:.4f} <<<   (board best 67.72 | rank-1 72.19)")\n        log("=" * 70)\n        log(f"  total time: {time.time()-t_start:.0f}s")\n\n        # append to history\n        try:\n            with open(HISTORY, "a") as f:\n                ts = time.strftime("%Y-%m-%d %H:%M")\n                f.write(f"{ts}  {Path(ZIP_PATH).name:<40}  FINAL {final:8.4f}  "\n                        f"PSNR {means[\'psnr\']:.4f}  MSSSIM {means[\'ms_ssim\']:.4f}  "\n                        f"LPIPS {means[\'lpips\']:.4f}  DISTS {means[\'dists\']:.4f}  "\n                        f"bpp {avg_bpp:.6f}  n={m}  {\'OK\' if within else \'OVER\'}\\n")\n            log(f"  appended to {HISTORY}")\n        except Exception as e:\n            log(f"  (history write skipped: {e})")\n    return 0 if within else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
score_script = WORK / "score_zip_boardfaithful_ft8_downscale.py"
score_script.write_text(SCORE_SCRIPT_SOURCE, encoding="utf-8")
text = score_script.read_text(encoding="utf-8")
history_path = WORK / "score_history_boardfaithful.txt"
text = re.sub(r'^ZIP_PATH\s*=.*$', f'ZIP_PATH   = r"{FINAL_ZIP}"', text, flags=re.M)
text = re.sub(r'^GT_DIR\s*=.*$', f'GT_DIR     = r"{INPUT_DIR}"', text, flags=re.M)
text = re.sub(r'^DEVICE\s*=.*$', f'DEVICE     = "{DEVICE}"', text, flags=re.M)
text = re.sub(r'^HISTORY\s*=.*$', f'HISTORY    = r"{history_path}"', text, flags=re.M)
score_script.write_text(text, encoding="utf-8")

if RUN_BOARD_FAITHFUL_SCORE:
    rc = run([sys.executable, score_script], cwd=WORK, check=False)
    print("Board-faithful scorer exit code:", rc)
else:
    print("Board-faithful scoring skipped.")
